In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn.utils.rnn import pad_sequence
from utils.helpers import *
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

device = find_backend()

Currently using:  mps


In [4]:
import pandas as pd

splits = {'train': 'Personality Datasets - Reddit/train_set.csv', 'validation': 'Personality Datasets - Reddit/val_set.csv', 'test': 'Personality Datasets - Reddit/eval_set.csv'}
train = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["train"])
test = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["test"])
validation = pd.read_csv("hf://datasets/Fatima0923/Automated-Personality-Prediction/" + splits["validation"])

In [5]:
p_type_names = ['agreeableness', 'openness', 'conscientiousness','extraversion', 'neuroticism']
train["personality"] = train[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
validation["personality"] = validation[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
test["personality"] = test[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)


/var/folders/l4/tfnrkckd1mj12_lf97sckpfc0000gn/T/ipykernel_70480/212605137.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  train["personality"] = train[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
/var/folders/l4/tfnrkckd1mj12_lf97sckpfc0000gn/T/ipykernel_70480/212605137.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  validation["personality"] = validation[p_type_names].apply(lambda x: torch.tensor(x), axis = 1)
/var/folders/l4/tfnrkckd1mj12_lf97sckpfc0000gn/T/ipykernel_70480/212605137.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer ke

In [6]:
from transformers import AutoTokenizer, DistilBertModel
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
# teacher_model = DistilBertModel.from_pretrained("distilbert-base-uncased")

To use data.metrics please install scikit-learn. See https://scikit-learn.org/stable/index.html


In [7]:
from models import encoder_decoder, decoder_only

model_type = "encoder_decoder" 
# model_type = "decoder_only"

vocab_size = tokenizer.vocab_size
hidden_dim=128
num_heads=2
dim_feedforward=2048
num_layers_enc=2
num_layers_dec=2
dropout=0.1
max_length=128
p_tags=5
ignore_index = -1

if model_type == "encoder_decoder":
    model =  encoder_decoder.EncoderDecoder(vocab_size,device, hidden_dim, num_heads, dim_feedforward, num_layers_enc, num_layers_dec, dropout,
                                            max_length, p_tags, ignore_index)

model = model.to(device)

In [8]:
input_test = torch.tensor(tokenizer.encode("This is an example")).unsqueeze(0).to(device)
output_test = torch.tensor(tokenizer.encode("This is output example")).unsqueeze(0).to(device)
personality_test = torch.rand(1,5).to(device)

input_test, output_test, personality_test

(tensor([[2023, 2003, 2019, 2742]], device='mps:0'),
 tensor([[2023, 2003, 6434, 2742]], device='mps:0'),
 tensor([[0.8949, 0.2638, 0.2396, 0.3462, 0.0477]], device='mps:0'))

In [9]:
out = model.forward(input_test, output_test, personality_test)
out.shape

/opt/homebrew/Caskroom/miniforge/base/envs/personality_llm/lib/python3.12/site-packages/torch/nn/functional.py:5962: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  warnings.warn(


torch.Size([1, 4, 30522])

In [18]:
model.generate_text(input_test, personality_test, tokenizer)

'[CLS] nearestleafwani banks clinchederal obtained gloss charge jan receive varietymu 143 westminsterctions evolving deformation person [unused645] 86 conrad secretsbay contestantsl supported seasoned beamedampwani dow silent homestead benefited person 505 liang reverted athletics prove gut mapped pastureposed azores 清 [unused926] buffy [unused800] person beau ticket material fate person ″dium crack directoreum nicky discussing nervously discover 09lateral healy palma rates contexts exeter cradle ʋ madeline negotiating secretsbay 1905pit華 [unused153] [unused289] montyein sylvie moves rouen vip rajasthan crossroads usl 313 toilets lds barbarians closest fools brunswick kun beverage piccolo caden stairsgement belmont fascination ر [unused645] glover colleen tears eaton mix europeanpit whites inflammationsat herds hears deficit hears conversations tilting olympiad catalan'

In [25]:
sample_obj = PersonalityTextDataset(train.sample(5), tokenizer, max_length, tokenizer.cls_token_id)

ValueError: Input is not valid. Should be a string, a list/tuple of strings or a list/tuple of integers.